## 策略评价benchmark

一个资产的return时间序列记作$[r_1,r_2,...,r_n]$，分布记为$R(\bar r,\sigma_r)$一个策略在每个时刻的position是$[p_1,p_2,...,p_n]$，分布记为$P(\bar{p}, \sigma_p)$。问题是如何评价这个策略是否显著有效。比如在一个普遍上涨（或下跌）的资产中，如何评价策略？

评价的benchmark可以定为完全随机的position（P与R的corr=0）下的收益分布,随机position分布的均值和方差与策略一致，即$[r_1 p_1,r_2 p_2,...,r_n p_n]$ 

那么比较策略均值收益率，即$B = \frac{1}{N} \sum_i p_i r_i$，那么$B$的分布为

$$
\begin{aligned}

E(B) &= \frac{1}{N}E(\sum_i p_i r_i) = \bar p  \bar r \\
\\
Var(B) &= \frac{1}{N^2} Var(\sum_i p_i r_i) = \frac{1}{N^2} \sum_i Var(p_ir_i) \\ 
       &= \frac{1}{N} ( Var(R)Var(P) + Var(R)E(P)^2 + Var(P)E(R)^2) \\
       &= \frac{1}{N} (\sigma_r^2 \sigma_p^2 + \sigma_r^2 \bar p^2 + \sigma_p^2 \bar r^2 ) \\
\end{aligned}

$$


即benchmark收益率均值的分布为$B(\bar r \bar p, \frac{1}{N}(\sigma_r^2 \sigma_p^2 + \sigma_r^2 \bar p^2 + \sigma_p^2 \bar r^2))$

由此可以计算出p-value来作为显著性的指标

几种情况的讨论

1. $\bar p = 0$，即策略平均持仓为零

此时的分布为$B(0, \frac{1}{N}(\sigma_r^2 \sigma_p^2 + \sigma_p^2 \bar r^2))$，即要判断策略是否显著偏离0，方差为$\frac{1}{N}(\sigma_r^2 \sigma_p^2 + \sigma_p^2 \bar r^2)$，持仓的杠杆越高，需要偏离的越大

2. $\bar r$ 显著偏离0，比如普遍上涨，$\bar r > 0$

此时的分布为 $B(\bar r \bar p, \frac{1}{N} (\sigma_r^2 \sigma_p^2 + \sigma_r^2 \bar p^2 + \sigma_p^2 \bar r^2))$，即要判断策略return是否显著偏离 平均持仓的return，对应的方差因为有第三项也会偏大

3. $\bar r = 0$，即资产涨跌较小，均值回归的情况

此时的分布为 $B(0, \frac{1}{N}(\sigma_r^2 \sigma_p^2 + \sigma_r^2 \bar p^2))$，即要判断策略return是否显著偏离0

计算代码如下

```python
#%% assessment: p-values

import pandas as pd
import numpy as np
from scipy import stats

def cal_pvalue(pos, rs):
    N = len(pos)
    mu = pos.mean() * rs.mean()
    var = (rs.var()*pos.var() + rs.var()*(pos.mean()**2) + pos.var()*(rs.mean()**2))/N
    sigma = var**0.5
    dist = stats.norm(loc=mu, scale=sigma)
    mn_s = (pos*rs).mean()
    p = 1 - dist.cdf(mn_s)
    return p
```

## 估算持仓时常

return的时间序列$[r_1,r_2,...,r_n]$满足分布$N(0,\sigma)$，假如某个策略胜率为$p$,交易成本为$cost$，估算最少持有多久，盈利才能覆盖交易成本？

假如持仓周期为$N$，设策略收益为$R$，则策略收益分布满足$N(0,\sqrt N \sigma)$，在胜率为p的条件下有

$$
\begin{aligned}
E(R) &= \int_{-\infty}^0 (1-p)P(R)RdR - \int_{-\infty}^0pP(R)RdR + \int_0^\infty pP(R)RdR - \int_0^\infty(1-p)P(R)RdR \\
     &= \int_0^\infty(4p-2)P(R)RdR \\
\end{aligned}
$$

对于正态分布而言，$\int_0^\infty P(R)RdR = \frac{\sqrt N \sigma}{2} \sqrt{\frac{2}{\pi}}$，带入有

$$
\begin{aligned}
E(R) &= (2p-1)\sqrt N \sigma \sqrt{\frac{2}{\pi}} > cost \\

N &>(\frac{cost}{\sigma \sqrt \frac{2}{\pi}(2p-1)})^2
\end{aligned}
$$